# Berlin EmoDB -> spectrogrammes RGB + CSV de metadonnees

Telecharge les wav d'EmoDB, fusionne les CSV audformat (db_files, db_emotion,
db_speaker, db_emotion_categories_*) pour recuperer toutes les infos par fichier,
puis convertit chaque wav en image RGB de mel-spectrogramme.

Les CSV audformat sont attendus dans data/csv/ (a placer manuellement).

Pretraitement : 16 kHz mono, fenetre de 3 s, trames de 25 ms toutes les 10 ms,
64 bandes mel en dB, standardisees (z-score par extrait).

In [ ]:
# deps (deja installees dans .venv, selectionner le kernel "Python (CNN_RO11 .venv)")
%pip install -q librosa matplotlib numpy pandas soundfile pillow nbstripout

# filtre git : retire les sorties et la metadata des .ipynb commites.
# A relancer une fois par clone (le filtre vit dans .git/config, qui n'est pas suivi).
import sys
!{sys.executable} -m nbstripout --install --attributes .gitattributes

In [ ]:
import io, zipfile, urllib.request
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from matplotlib import cm
from PIL import Image

URL = "https://zenodo.org/api/records/17651657/files-archive"
DATA_DIR = Path("data")
RAW = DATA_DIR / "emodb_2.0"
CSV_DIR = Path("data/csv")
OUT = Path("data/spectrograms")
LABELS_CSV = Path("data/labels.csv")

DATASET_NAME = "EmoDB"

# colonnes communes a toutes les bases (union des taxonomies)
EMOTIONS_ALL = [
    "anger", "boredom", "calm", "disgust", "fear",
    "happiness", "neutral", "sadness", "surprise",
]
EMODB_EMOTIONS = {"anger", "boredom", "disgust", "fear", "happiness", "neutral", "sadness"}

In [ ]:
# 1. telechargement + extraction (l'archive contient elle-meme des zip)
if not RAW.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("downloading...")
    data = urllib.request.urlopen(URL).read()
    zipfile.ZipFile(io.BytesIO(data)).extractall(DATA_DIR)

    inner_zips = list(DATA_DIR.rglob("*.zip"))
    while inner_zips:
        for inner in inner_zips:
            zipfile.ZipFile(inner).extractall(inner.parent)
            inner.unlink()
        inner_zips = list(DATA_DIR.rglob("*.zip"))

wavs = sorted(p for p in RAW.rglob("*.wav") if p.parent.name == "wav")
print(len(wavs), "fichiers wav")

In [ ]:
# 2. fusion des CSV audformat -> 1 CSV (accumule sur plusieurs bases)
import pandas as pd

FILES_CSV = "db.files.csv"
SPEAKER_CSV = "db.speaker.csv"
EMOTION_CSV = "db.emotion.csv"

def find_csv(name):
    stem, ext = name.rsplit(".", 1)
    for cand in {name, stem.replace(".", "_") + "." + ext}:
        matches = list(RAW.rglob(cand))
        if matches:
            return matches[0]
    raise FileNotFoundError(name)

files = pd.read_csv(find_csv(FILES_CSV))
speaker = pd.read_csv(find_csv(SPEAKER_CSV))
emotion = pd.read_csv(find_csv(EMOTION_CSV))

df = files.merge(speaker, on="speaker", how="left")
df = df.merge(emotion, on="file", how="left")

df["subdir"] = df["file"].apply(lambda p: str(Path(p).parent))
df["file"] = df["file"].apply(lambda p: Path(p).name)
df["duration"] = [round(sf.info(RAW / s / f).duration, 3) for s, f in zip(df["subdir"], df["file"])]
df.insert(0, "dataset", DATASET_NAME)
df.insert(0, "root", str(RAW))

if LABELS_CSV.exists():
    existing = pd.read_csv(LABELS_CSV, na_values=["None"])
    dup = existing.merge(df[["root", "file"]], on=["root", "file"], how="inner")
    if len(dup):
        raise ValueError(f"{len(dup)} fichiers de {DATASET_NAME} deja presents dans {LABELS_CSV}")
    same = existing[existing["dataset"] == DATASET_NAME]
    prior_speaker = dict(zip(same["initial_speaker"], same["speaker"]))
    prior_transcript = dict(zip(same["initial_transcript"], same["transcript"]))
    next_speaker = int(existing["speaker"].max()) + 1
    next_transcript = int(existing["transcript"].max()) + 1
else:
    existing = None
    prior_speaker = {}
    prior_transcript = {}
    next_speaker = 1
    next_transcript = 1

df["initial_speaker"] = df["speaker"]
speaker_map = {}
for code in sorted(df["speaker"].unique()):
    if code in prior_speaker:
        speaker_map[code] = prior_speaker[code]
    else:
        speaker_map[code] = next_speaker
        next_speaker += 1
df["speaker"] = df["speaker"].map(speaker_map)

df["initial_transcript"] = df["transcription"]
transcript_map = {}
for code in sorted(df["transcription"].unique()):
    if code in prior_transcript:
        transcript_map[code] = prior_transcript[code]
    else:
        transcript_map[code] = next_transcript
        next_transcript += 1
df["transcript"] = df["transcription"].map(transcript_map)
df = df.drop(columns=["transcription"])

for emo in EMOTIONS_ALL:
    df[emo] = (df["emotion"] == emo).astype(int) if emo in EMODB_EMOTIONS else None
df = df.drop(columns=["emotion"])

cols = ["root", "dataset", "subdir", "file", "duration", "speaker", "initial_speaker",
        "age", "gender", "language", "emotion.confidence", "emotion.naturalness",
        "transcript", "initial_transcript"] + EMOTIONS_ALL
df = df[cols]

if existing is not None:
    df = pd.concat([existing, df], ignore_index=True)

df.to_csv(LABELS_CSV, index=False, na_rep="None")
print(len(df), "lignes dans", LABELS_CSV)

In [ ]:
# 3. wav -> segments de 3s -> mel-spectrogramme -> image RGB (accumule sur plusieurs bases)
SR = 16000
DURATION = 3.0
MIN_SEG = 1.0
N_FFT = int(0.025 * SR)
HOP = int(0.010 * SR)
N_MELS = 64
N_FRAMES = int(DURATION * SR / HOP)
CHUNK = int(DURATION * SR)
MIN_SAMPLES = int(MIN_SEG * SR)

SPEC_CSV = OUT / "labels.csv"
label_by_file = dict(zip(df["file"], df[EMOTIONS_ALL].fillna(0).idxmax(axis=1)))
meta = df.set_index("file").to_dict(orient="index")

spec_existing = pd.read_csv(SPEC_CSV, na_values=["None"]) if SPEC_CSV.exists() else None
done = set(zip(spec_existing["root"], spec_existing["base_file"])) if spec_existing is not None else set()

def mel_to_rgb(y):
    y = librosa.util.fix_length(y, size=CHUNK)
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, win_length=N_FFT, n_mels=N_MELS
    )
    db = librosa.power_to_db(mel)[:, :N_FRAMES]
    z = (db - db.mean()) / (db.std() + 1e-9)
    norm = np.clip(z, -3, 3) / 6 + 0.5
    rgb = (cm.viridis(norm[::-1])[..., :3] * 255).astype(np.uint8)
    return Image.fromarray(rgb)

rows = []
for wav in wavs:
    if (str(RAW), wav.name) in done:
        continue
    y, _ = librosa.load(wav, sr=SR, mono=True)
    label = label_by_file[wav.name]

    for n, start in enumerate(range(0, len(y), CHUNK)):
        seg = y[start:start + CHUNK]
        if len(seg) < MIN_SAMPLES:
            continue
        name = f"{wav.stem}_{n}"
        dest = OUT / label
        dest.mkdir(parents=True, exist_ok=True)
        mel_to_rgb(seg).save(dest / f"{name}.png")
        rows.append({
            "spectrogram": f"{label}/{name}.png",
            "base_file": wav.name,
            "chunk": n,
            **meta[wav.name],
            "duration": round(sf.info(wav).duration, 3),
        })

new_spec = pd.DataFrame(rows)
spec_df = pd.concat([spec_existing, new_spec], ignore_index=True) if spec_existing is not None else new_spec
spec_df.to_csv(SPEC_CSV, index=False, na_rep="None")
print(len(rows), "nouveaux spectrogrammes,", len(spec_df), "total dans", SPEC_CSV)

In [ ]:
# 4. verification rapide
for d in sorted(OUT.iterdir()):
    print(d.name, len(list(d.glob("*.png"))))

display(Image.open(next(OUT.rglob("*.png"))))
display(df.head())